# Full RAG + Uncertainty Estimation Pipeline
### IR Project — Arnes HPC Jupyter Environment

**Model:** Qwen2.5-7B-Instruct

**Pipeline:**
- **Retriever:** BM25 + Cross-Encoder Reranker (ms-marco-MiniLM-L6-v2)
- **Generator:** Qwen2.5-7B-Instruct
- **Evaluation:** SAFE (using local Lucene index)

**Uncertainty Estimation Methods:**
- **MARS (White-box):** Using TruthTorchLM
- **Eccentricity (Black-box):** Using TruthTorchLM
- **BONUS - Semantic Entropy (Hybrid):** Cluster-based semantic uncertainty

## 1. Environment Configuration

In [ ]:
# ============================================================================
# CELL 1: HuggingFace Cache Configuration (MUST BE FIRST CELL)
# ============================================================================
import os

# HuggingFace cache redirection - adjust paths for your cluster
os.environ["HF_HOME"] = "/d/hpc/projects/FRI/ma76193/hf_home"
os.environ["HF_DATASETS_CACHE"] = "/d/hpc/projects/FRI/ma76193/hf_datasets"
# Note: TRANSFORMERS_CACHE is deprecated, but kept for backwards compatibility
os.environ["TRANSFORMERS_CACHE"] = "/d/hpc/projects/FRI/ma76193/hf_transformers"

for path in [os.environ["HF_HOME"], os.environ["HF_DATASETS_CACHE"], os.environ["TRANSFORMERS_CACHE"]]:
    os.makedirs(path, exist_ok=True)

print(f"HF_HOME = {os.environ['HF_HOME']}")
print(f"HF_DATASETS_CACHE = {os.environ['HF_DATASETS_CACHE']}")
print(f"TRANSFORMERS_CACHE = {os.environ['TRANSFORMERS_CACHE']}")

In [ ]:
# ============================================================================
# CELL 2: Java Configuration & Logging Setup
# ============================================================================
import os, sys, json, time, random, logging, threading, subprocess
from pathlib import Path
from datetime import datetime

# Java configuration for Pyserini
JAVA = "/d/hpc/projects/FRI/ma76193/miniconda3/envs/ragpy311/jdk-21.0.1+12"
os.environ["JAVA_HOME"] = JAVA
os.environ["JVM_PATH"] = f"{JAVA}/lib/server/libjvm.so"
os.environ["LD_LIBRARY_PATH"] = f"{JAVA}/lib/server:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["PATH"] = f"{JAVA}/bin:" + os.environ["PATH"]

print(f"JAVA_HOME = {os.environ['JAVA_HOME']}")
!java -version

import torch

# Logging
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / "full_notebook.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler(LOG_FILE, mode="a", encoding="utf-8"), logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("nb")

def banner(msg):
    log.info("=" * 80)
    log.info(f"*** {msg} ***")
    log.info("=" * 80)

In [ ]:
# ============================================================================
# CELL 3: Heartbeat & Environment Optimization
# ============================================================================
_stop_hb = threading.Event()

def _heartbeat(period=30):
    while not _stop_hb.is_set():
        log.info("[HEARTBEAT] Notebook alive...")
        time.sleep(period)

hb = threading.Thread(target=_heartbeat, daemon=True)
hb.start()

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
os.environ.setdefault("JAVA_TOOL_OPTIONS", "-Xms1g -Xmx8g")

log.info(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        log.info(f"GPU[{i}] {torch.cuda.get_device_name(i)}")

In [ ]:
# ============================================================================
# CELL 4: Directory Setup
# ============================================================================
FULL_SHARDS_DIR = Path("data/wiki18/shards_full")
FULL_INDEX_DIR = Path("index/bm25_full")
RUNS = Path("runs")
RUNS.mkdir(exist_ok=True)

RETR_DIR = RUNS / "retrieval"
ANS_DIR = RUNS / "answers"
UE_DIR = RUNS / "ue"
REPORTS_DIR = Path("reports")

for p in [RETR_DIR, ANS_DIR, UE_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

banner("Directories prepared")

## 2. Data Preparation: Wiki-18 Corpus

In [ ]:
# ============================================================================
# CELL 5: Download Wiki-18 Dataset
# ============================================================================
from huggingface_hub import snapshot_download
import gzip

HF_DATA_REPO = "PeterJinGo/wiki-18-corpus"

def download_wiki18():
    banner("DOWNLOAD WIKI-18")
    path = snapshot_download(repo_id=HF_DATA_REPO, repo_type="dataset", allow_patterns=["*.jsonl.gz"], local_files_only=False)
    candidates = list(Path(path).rglob("*.jsonl.gz"))
    if not candidates:
        raise RuntimeError("No wiki18 .jsonl.gz found")
    return candidates[0]

wiki_gz = download_wiki18()
log.info(f"wiki gz: {wiki_gz}")

In [ ]:
# ============================================================================
# CELL 6: Shard Wiki-18 & Build BM25 Index
# ============================================================================
def shard_wiki18(gz_path, out_dir, shard_size=100_000):
    banner("SHARDING WIKI-18")
    out_dir.mkdir(parents=True, exist_ok=True)
    if list(out_dir.glob("*.jsonl")):
        log.info(f"Shards exist. Skipping.")
        return
    total, idx, written = 0, 0, 0
    out_f = open(out_dir / f"wiki18_full_{idx:03d}.jsonl", "w", encoding="utf-8")
    with gzip.open(gz_path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            try:
                obj = json.loads(line)
            except:
                continue
            text = obj.get("text") or obj.get("contents") or ""
            if not text.strip():
                continue
            rec = {"id": obj.get("id") or obj.get("page_id") or str(total), "contents": text.strip()}
            out_f.write(json.dumps(rec) + "\n")
            total += 1
            written += 1
            if written >= shard_size:
                out_f.close()
                idx += 1
                written = 0
                out_f = open(out_dir / f"wiki18_full_{idx:03d}.jsonl", "w", encoding="utf-8")
    out_f.close()
    log.info(f"Sharding complete. Total: {total}")

def build_index(input_dir, index_dir, threads=8):
    banner("BM25 INDEX BUILD")
    if index_dir.exists() and any(index_dir.iterdir()):
        log.info("Index exists. Skipping.")
        return
    cmd = [sys.executable, "-m", "pyserini.index.lucene", "--collection", "JsonCollection",
           "--input", str(input_dir), "--index", str(index_dir), "--generator", "DefaultLuceneDocumentGenerator",
           "--threads", str(threads), "--storePositions", "--storeDocvectors", "--storeRaw"]
    subprocess.run(cmd, check=True)

shard_wiki18(wiki_gz, FULL_SHARDS_DIR)
build_index(FULL_SHARDS_DIR, FULL_INDEX_DIR)

## 3. Query Sampling

In [ ]:
# ============================================================================
# CELL 7: Sample Queries
# ============================================================================
def extract_query_text(obj):
    for k in ["query", "question", "prompt", "instruction", "text", "claim"]:
        if k in obj and isinstance(obj[k], str) and obj[k].strip():
            return obj[k].strip()
    for v in obj.values():
        if isinstance(v, str) and v.strip():
            return v.strip()
    return None

def sample_queries(src_path, out_path, n, seed):
    banner("SAMPLE QUERIES")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists():
        log.info(f"Exists: {out_path}")
        return out_path
    items = []
    with open(src_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            try:
                obj = json.loads(line)
            except:
                continue
            q = extract_query_text(obj)
            if q:
                qid = str(obj.get("id") or obj.get("_id") or f"s{i:08d}")
                items.append((qid, q))
    random.Random(seed).shuffle(items)
    with open(out_path, "w", encoding="utf-8") as f:
        for i, (qid, q) in enumerate(items[:n], 1):
            f.write(json.dumps({"id": f"b{i:03d}", "orig_id": qid, "query": q}, ensure_ascii=False) + "\n")
    log.info(f"Wrote {n} queries -> {out_path}")
    return out_path

QUERY_SRC = Path("data/queries/factscore_bio.jsonl")
SAMPLED_QUERIES = Path("data/queries/notebook.seed1337.jsonl")
sample_queries(QUERY_SRC, SAMPLED_QUERIES, n=50, seed=1337)

## 4. Retrieval: BM25 + Cross-Encoder Reranking

In [ ]:
# ============================================================================
# CELL 8: BM25 + Cross-Encoder Reranking
# ============================================================================
from pyserini.search.lucene import LuceneSearcher
from sentence_transformers import CrossEncoder

def get_doc_text(raw):
    try:
        return json.loads(raw).get("contents", "").strip()
    except:
        return str(raw).strip()

def retrieve_rerank(queries_path, index_dir, out_path, k_first=1000, k_keep=3,
                    ce_model="cross-encoder/ms-marco-MiniLM-L6-v2", batch_size=64):
    banner("BM25 + CROSS-ENCODER RERANK")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    done = {}
    if out_path.exists():
        with open(out_path, "r") as f:
            for line in f:
                try:
                    ex = json.loads(line)
                    done[ex["id"]] = ex
                except:
                    pass
        log.info(f"Resume: {len(done)} done")
    try:
        ce = CrossEncoder(ce_model, device="cuda" if torch.cuda.is_available() else "cpu")
    except:
        ce = None
    searcher = LuceneSearcher(str(index_dir))
    searcher.set_bm25(k1=0.9, b=0.4)
    new = 0
    with open(queries_path, "r") as fin, open(out_path, "a") as fout:
        for line in fin:
            ex = json.loads(line)
            qid, query = ex["id"], ex["query"]
            if qid in done:
                continue
            hits = searcher.search(query, k_first)
            if not hits:
                fout.write(json.dumps({"id": qid, "query": query, "docs": []}) + "\n")
                new += 1
                continue
            candidates = [{"docid": h.docid, "raw": searcher.doc(h.docid).raw(),
                          "bm25": float(h.score), "text": get_doc_text(searcher.doc(h.docid).raw())} for h in hits]
            if ce:
                try:
                    scores = ce.predict([(query, c["text"]) for c in candidates], batch_size=batch_size)
                    for c, s in zip(candidates, scores):
                        c["ce"] = float(s)
                    candidates.sort(key=lambda x: x.get("ce", -1e9), reverse=True)
                except:
                    candidates.sort(key=lambda x: x["bm25"], reverse=True)
            else:
                candidates.sort(key=lambda x: x["bm25"], reverse=True)
            final_docs = [{"docid": c["docid"], "raw": c["raw"], "bm25": c["bm25"], "ce": c.get("ce")} for c in candidates[:k_keep]]
            fout.write(json.dumps({"id": qid, "query": query, "docs": final_docs}) + "\n")
            new += 1
            log.info(f"[{qid}] {len(final_docs)} docs")
    log.info(f"Rerank done: {new} new")
    return out_path

RETR_FILE = RETR_DIR / "notebook.seed1337.rerank3.jsonl"
retrieve_rerank(SAMPLED_QUERIES, FULL_INDEX_DIR, RETR_FILE, k_first=1000, k_keep=3)

## 5. LLM Loading: Qwen2.5-7B-Instruct

In [ ]:
# ============================================================================
# CELL 9: Load Qwen2.5-7B-Instruct with Smart Caching
# ============================================================================
from transformers import AutoTokenizer, AutoModelForCausalLM
import shutil

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

def parse_docs_for_prompt(docs):
    texts = []
    for i, d in enumerate(docs, 1):
        try:
            text = json.loads(d["raw"]).get("contents", "")
        except:
            text = d.get("raw", "")
        text = " ".join(text.split())[:1200]
        texts.append(f"[d{i}] {text}")
    return "\n".join(texts)

def build_prompt(query, docs, strict=True):
    policy = "Answer only using the provided documents. If the documents do not contain the answer, say you don't know." if strict else "Prefer the provided documents; if insufficient, you may use general knowledge."
    return f"You are a careful assistant. {policy}\n\nQuestion: {query}\n\nDocuments:\n{parse_docs_for_prompt(docs)}\n\nAnswer:"

def load_llm_smart(model_id, force_download=False):
    """
    Smart model loading:
    1. Try to load from local cache first
    2. If fails, download fresh copy
    """
    banner(f"LOADING LLM: {model_id}")
    
    hf_home = os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
    model_cache_name = model_id.replace("/", "--")
    model_cache_path = Path(hf_home) / "hub" / f"models--{model_cache_name}"
    
    def try_load(local_files_only=False):
        tok = AutoTokenizer.from_pretrained(
            model_id, 
            use_fast=True, 
            trust_remote_code=True,
            local_files_only=local_files_only
        )
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        
        if torch.cuda.is_available():
            mdl = AutoModelForCausalLM.from_pretrained(
                model_id, 
                device_map="auto", 
                torch_dtype=torch.bfloat16, 
                trust_remote_code=True,
                local_files_only=local_files_only
            )
            device = "cuda"
        else:
            mdl = AutoModelForCausalLM.from_pretrained(
                model_id, 
                trust_remote_code=True,
                local_files_only=local_files_only
            )
            device = "cpu"
        return tok, mdl, device
    
    # Step 1: Try loading from local cache
    if not force_download and model_cache_path.exists():
        log.info(f"Found local cache at {model_cache_path}, attempting to load...")
        try:
            tok, mdl, device = try_load(local_files_only=True)
            log.info(f"✓ Successfully loaded {model_id} from local cache on {device}")
            return tok, mdl, device
        except Exception as e:
            log.warning(f"Local cache load failed: {e}")
            log.info("Cache may be corrupted. Will download fresh copy...")
            try:
                shutil.rmtree(model_cache_path)
                log.info(f"Removed corrupted cache at {model_cache_path}")
            except Exception as rm_err:
                log.warning(f"Could not remove cache: {rm_err}")
    
    # Step 2: Download fresh copy
    log.info(f"Downloading {model_id} from HuggingFace Hub...")
    try:
        tok, mdl, device = try_load(local_files_only=False)
        log.info(f"✓ Successfully downloaded and loaded {model_id} on {device}")
        return tok, mdl, device
    except Exception as e:
        log.error(f"Failed to download model: {e}")
        raise RuntimeError(f"Could not load model {model_id}: {e}")

# Load Qwen
llm_tok, llm_mdl, llm_device = load_llm_smart(MODEL_ID)
print(f"\n[INFO] Using: {MODEL_ID} on {llm_device}")

## 6. Answer Generation

In [ ]:
# ============================================================================
# CELL 10: Generate RAG Answers
# ============================================================================
def generate_answers(retr_path, out_path, tok, mdl, device, model_name, max_new_tokens=256, strict=True):
    banner(f"GENERATE ANSWERS: {model_name}")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    done = set()
    if out_path.exists():
        with open(out_path, "r") as f:
            for line in f:
                try:
                    done.add(json.loads(line)["id"])
                except:
                    pass
        log.info(f"Resume: {len(done)} done")
    new = 0
    with open(retr_path, "r") as fin, open(out_path, "a") as fout:
        for line in fin:
            ex = json.loads(line)
            qid, q, docs = ex["id"], ex["query"], ex["docs"]
            if qid in done:
                continue
            prompt = build_prompt(q, docs, strict=strict)
            enc = tok(prompt, return_tensors="pt", truncation=True, max_length=2048)
            if device == "cuda":
                enc = {k: v.cuda() for k, v in enc.items()}
            with torch.no_grad():
                out = mdl.generate(**enc, do_sample=False, max_new_tokens=max_new_tokens,
                                   eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id or tok.eos_token_id)
            full_text = tok.decode(out[0], skip_special_tokens=True)
            answer = full_text.split("Answer:", 1)[-1].strip() if "Answer:" in full_text else full_text.strip()
            record = {"id": qid, "query": q, "answer": answer, "docs": docs,
                     "meta": {"ts": datetime.now().isoformat(timespec="seconds"), "model": model_name}}
            fout.write(json.dumps(record, ensure_ascii=False) + "\n")
            new += 1
            log.info(f"[{qid}] len={len(answer)}")
    log.info(f"Done: {new} new")
    return out_path

ANSWERS_FILE = ANS_DIR / "notebook.seed1337.qwen7b.jsonl"
generate_answers(RETR_FILE, ANSWERS_FILE, llm_tok, llm_mdl, llm_device, MODEL_ID)

## 7. SAFE Evaluation

In [ ]:
# ============================================================================
# CELL 11: SAFE Evaluation (using installed TruthTorchLM package)
# ============================================================================
import pandas as pd

def load_safe_evaluator():
    """Load SAFE evaluator from installed TruthTorchLM package."""
    banner("INIT SAFE")
    
    # Import from installed package (site-packages)
    try:
        from TruthTorchLM.long_form_generation.evaluators.eval_claim import ClaimEvaluator
        log.info("Loaded ClaimEvaluator from installed TruthTorchLM")
    except ImportError as e:
        log.error(f"ClaimEvaluator not found: {e}")
        return None
    
    # Use already-loaded Qwen model
    return ClaimEvaluator(
        rater_model=llm_mdl, 
        rater_tokenizer=llm_tok, 
        lucene_index_dir=str(FULL_INDEX_DIR), 
        max_steps=3, 
        max_retries=3, 
        bm25_k=3
    )

def run_safe(answers_file, out_jsonl, out_csv, evaluator):
    banner("RUN SAFE")
    if evaluator is None:
        log.error("SAFE evaluator not available")
        return
    
    # Resume support
    done = set()
    rows = []
    if out_jsonl.exists():
        with open(out_jsonl, "r") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    done.add(obj["id"])
                    rows.append(obj)
                except:
                    pass
        log.info(f"Resume: {len(done)} done")
    
    with open(answers_file, "r") as fin, open(out_jsonl, "a") as fout:
        for line in fin:
            ex = json.loads(line)
            if ex["id"] in done:
                continue
            log.info(f"[SAFE] {ex['id']}")
            try:
                res = evaluator(ex["answer"])
                row = {
                    "id": ex["id"], 
                    "safe_score": res.get("answer"), 
                    "safe_response": res.get("response", ""), 
                    "safe_details": res.get("search_details", [])
                }
            except Exception as e:
                log.warning(f"SAFE error for {ex['id']}: {e}")
                row = {"id": ex["id"], "safe_score": None, "safe_response": str(e), "safe_details": []}
            rows.append(row)
            fout.write(json.dumps(row) + "\n")
            fout.flush()
    
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    log.info(f"SAFE done: {out_jsonl}")

safe_eval = load_safe_evaluator()
SAFE_JSONL = UE_DIR / "notebook.seed1337.safe.jsonl"
SAFE_CSV = UE_DIR / "notebook.seed1337.safe.csv"
run_safe(ANSWERS_FILE, SAFE_JSONL, SAFE_CSV, safe_eval)

## 8. MARS (White-box UE) - TruthTorchLM

In [ ]:
# ============================================================================
# CELL 12: MARS Uncertainty Estimation using TruthTorchLM
# ============================================================================
import TruthTorchLM as ttlm

def run_mars(answers_file, out_jsonl, out_csv):
    banner("MARS (TruthTorchLM)")
    
    # Initialize MARS from TruthTorchLM
    mars = ttlm.truth_methods.MARS()
    
    # Resume support
    done = set()
    rows = []
    if out_jsonl.exists():
        with open(out_jsonl, "r") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    done.add(obj["id"])
                    rows.append(obj)
                except:
                    pass
        log.info(f"Resume: {len(done)} done")
    
    with open(answers_file, "r") as fin, open(out_jsonl, "a") as fout:
        for line in fin:
            ex = json.loads(line)
            if ex["id"] in done:
                continue
            
            qid, q, ans, docs = ex["id"], ex["query"], ex["answer"], ex["docs"]
            context = "\n".join([get_doc_text(d.get("raw", "")) for d in docs])
            
            try:
                # Build chat messages for TruthTorchLM
                chat = [
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": f"Context:\n{context[:2000]}\n\nQuestion: {q}"}
                ]
                
                # Generate with truth value
                output = ttlm.generate_with_truth_value(
                    model=llm_mdl,
                    tokenizer=llm_tok,
                    messages=chat,
                    truth_methods=[mars],
                    max_new_tokens=256,
                    do_sample=False
                )
                score = output.get("truth_values", {}).get("MARS")
                if score is not None:
                    score = float(score)
                details = {"method": "truthtorchlm"}
            except Exception as e:
                log.warning(f"MARS failed for {qid}: {e}")
                score = None
                details = {"method": "failed", "error": str(e)}
            
            row = {"id": qid, "mars_score": score, "mars_details": details}
            rows.append(row)
            fout.write(json.dumps(row) + "\n")
            fout.flush()
            
            if score is not None:
                log.info(f"[{qid}] MARS={score:.4f}")
            else:
                log.info(f"[{qid}] MARS=None")
    
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    log.info(f"MARS done: {out_jsonl}")

MARS_JSONL = UE_DIR / "notebook.seed1337.mars.jsonl"
MARS_CSV = UE_DIR / "notebook.seed1337.mars.csv"
run_mars(ANSWERS_FILE, MARS_JSONL, MARS_CSV)

## 9. Eccentricity (Black-box UE) - TruthTorchLM

In [ ]:
# ============================================================================
# CELL 13: Eccentricity Uncertainty using TruthTorchLM
# ============================================================================
from sentence_transformers import SentenceTransformer
import numpy as np
import csv

def run_eccentricity(answers_file, out_jsonl, out_csv):
    banner("ECCENTRICITY (TruthTorchLM)")
    
    # Initialize Eccentricity from TruthTorchLM
    ecc = ttlm.truth_methods.EccentricityUncertainty()
    
    # Resume support
    done = set()
    rows = []
    if out_jsonl.exists():
        with open(out_jsonl, "r") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    done.add(obj["id"])
                    rows.append(obj)
                except:
                    pass
        log.info(f"Resume: {len(done)} done")
    
    with open(answers_file, "r") as fin, open(out_jsonl, "a") as fout:
        for line in fin:
            ex = json.loads(line)
            if ex["id"] in done:
                continue
            
            qid, q, ans, docs = ex["id"], ex["query"], ex["answer"], ex["docs"]
            context = "\n".join([get_doc_text(d.get("raw", "")) for d in docs])
            
            try:
                chat = [
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": f"Context:\n{context[:2000]}\n\nQuestion: {q}"}
                ]
                
                output = ttlm.generate_with_truth_value(
                    model=llm_mdl,
                    tokenizer=llm_tok,
                    messages=chat,
                    truth_methods=[ecc],
                    max_new_tokens=256,
                    do_sample=False
                )
                score = output.get("truth_values", {}).get("EccentricityUncertainty")
                if score is not None:
                    score = float(score)
            except Exception as e:
                log.warning(f"ECC failed for {qid}: {e}")
                score = None
            
            row = {"id": qid, "ecc": score, "ecc_z": None}
            rows.append(row)
            fout.write(json.dumps(row) + "\n")
            fout.flush()
            
            if score is not None:
                log.info(f"[{qid}] ECC={score:.4f}")
            else:
                log.info(f"[{qid}] ECC=None")
    
    # Compute z-scores
    vals = np.array([r["ecc"] for r in rows if r["ecc"] is not None])
    if len(vals) > 1:
        mu, sd = vals.mean(), max(vals.std(ddof=1), 1e-6)
        for r in rows:
            if r["ecc"] is not None:
                r["ecc_z"] = float((r["ecc"] - mu) / sd)
    
    # Rewrite with z-scores
    with open(out_jsonl, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    
    with open(out_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["id", "ecc", "ecc_z"])
        w.writeheader()
        w.writerows(rows)
    
    log.info(f"ECC done: {out_jsonl}")

ECC_JSONL = UE_DIR / "notebook.seed1337.ecc.jsonl"
ECC_CSV = UE_DIR / "notebook.seed1337.ecc.csv"
run_eccentricity(ANSWERS_FILE, ECC_JSONL, ECC_CSV)

## 10. BONUS: Semantic Entropy

A novel UE method that generates multiple answer variants and measures semantic diversity.
Higher entropy = more semantically diverse answers = higher uncertainty.

In [ ]:
# ============================================================================
# CELL 14: BONUS - Semantic Entropy
# ============================================================================
from sklearn.cluster import AgglomerativeClustering
from collections import Counter

def compute_semantic_entropy(query, docs, tok, mdl, embed_model, n_samples=5, temperature=0.7, max_new_tokens=128, sim_thresh=0.85):
    prompt = build_prompt(query, docs, strict=True)
    answers = []
    enc = tok(prompt, return_tensors="pt", truncation=True, max_length=1536)
    if torch.cuda.is_available():
        enc = {k: v.cuda() for k, v in enc.items()}
    for _ in range(n_samples):
        with torch.no_grad():
            out = mdl.generate(**enc, do_sample=True, temperature=temperature, top_p=0.9, max_new_tokens=max_new_tokens,
                              eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id or tok.eos_token_id)
        full_text = tok.decode(out[0], skip_special_tokens=True)
        ans = full_text.split("Answer:", 1)[-1].strip() if "Answer:" in full_text else full_text[len(prompt):].strip()
        if ans:
            answers.append(ans)
    if len(answers) < 2:
        return {"semantic_entropy": 0.0, "n_clusters": 1, "n_samples": len(answers)}
    embeddings = embed_model.encode(answers, normalize_embeddings=True)
    clustering = AgglomerativeClustering(n_clusters=None, distance_threshold=1-sim_thresh, metric='cosine', linkage='average')
    labels = clustering.fit_predict(embeddings)
    counts = Counter(labels)
    total = sum(counts.values())
    entropy = -sum((c/total) * np.log2(c/total) for c in counts.values() if c > 0)
    max_ent = np.log2(len(answers)) if len(answers) > 1 else 1.0
    return {"semantic_entropy": float(entropy / max_ent), "raw_entropy": float(entropy), "n_clusters": len(counts), "n_samples": len(answers)}

def run_semantic_entropy(answers_file, out_jsonl, out_csv, tok, mdl, n_samples=5, embed_model_name="sentence-transformers/all-MiniLM-L6-v2"):
    banner("SEMANTIC ENTROPY (BONUS)")
    embed_model = SentenceTransformer(embed_model_name, device="cuda" if torch.cuda.is_available() else "cpu")
    
    # Resume support
    done = set()
    rows = []
    if out_jsonl.exists():
        with open(out_jsonl, "r") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    done.add(obj["id"])
                    rows.append(obj)
                except:
                    pass
        log.info(f"Resume: {len(done)} done")
    
    with open(answers_file, "r") as fin, open(out_jsonl, "a") as fout:
        for line in fin:
            ex = json.loads(line)
            if ex["id"] in done:
                continue
            log.info(f"[SemEnt] {ex['id']}")
            try:
                res = compute_semantic_entropy(ex["query"], ex["docs"], tok, mdl, embed_model, n_samples)
                row = {"id": ex["id"], "semantic_entropy": res["semantic_entropy"], "raw_entropy": res.get("raw_entropy",0), "n_clusters": res["n_clusters"], "n_samples": res["n_samples"]}
            except Exception as e:
                log.warning(f"SemEnt failed for {ex['id']}: {e}")
                row = {"id": ex["id"], "semantic_entropy": None, "raw_entropy": None, "n_clusters": None, "n_samples": None}
            rows.append(row)
            fout.write(json.dumps(row) + "\n")
            fout.flush()
    
    # Compute z-scores
    vals = np.array([r["semantic_entropy"] for r in rows if r["semantic_entropy"] is not None])
    if len(vals) > 1:
        mu, sd = vals.mean(), max(vals.std(ddof=1), 1e-6)
        for r in rows:
            if r["semantic_entropy"] is not None:
                r["sem_ent_z"] = float((r["semantic_entropy"] - mu) / sd)
            else:
                r["sem_ent_z"] = None
    
    # Rewrite with z-scores
    with open(out_jsonl, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    log.info(f"SemEnt done: {out_jsonl}")

SEMENT_JSONL = UE_DIR / "notebook.seed1337.semantic_entropy.jsonl"
SEMENT_CSV = UE_DIR / "notebook.seed1337.semantic_entropy.csv"
run_semantic_entropy(ANSWERS_FILE, SEMENT_JSONL, SEMENT_CSV, llm_tok, llm_mdl, n_samples=5)

## 11. Final Report

In [ ]:
# ============================================================================
# CELL 15: Final Report with Correlations
# ============================================================================

def format_val(val, fmt=".3f"):
    """Helper function to safely format values with NaN handling."""
    if pd.notna(val):
        return f"{val:{fmt}}"
    return "N/A"

def write_final_report(queries_file, answers_file, mars_file, ecc_file, safe_file, sement_file, out_md, out_csv, corr_csv):
    banner("FINAL REPORT")
    
    def load_jsonl(p):
        d = {}
        if p.exists():
            with open(p, "r") as f:
                for line in f:
                    try:
                        obj = json.loads(line)
                        d[obj["id"]] = obj
                    except:
                        pass
        return d
    
    answers = load_jsonl(answers_file)
    mars = load_jsonl(mars_file)
    ecc = load_jsonl(ecc_file)
    safe = load_jsonl(safe_file)
    sement = load_jsonl(sement_file) if sement_file.exists() else {}
    
    log.info(f"Loaded: {len(answers)} answers, {len(mars)} MARS, {len(ecc)} ECC, {len(safe)} SAFE, {len(sement)} SemEnt")
    
    rows = []
    for qid, ans_data in answers.items():
        safe_label = safe.get(qid, {}).get("safe_score")
        correctness = 1.0 if safe_label == "Supported" else (0.0 if safe_label == "Not Supported" else np.nan)
        rows.append({
            "id": qid, 
            "query": ans_data.get("query", "")[:80], 
            "safe": safe_label, 
            "correctness": correctness,
            "mars": mars.get(qid, {}).get("mars_score"), 
            "ecc": ecc.get(qid, {}).get("ecc"),
            "ecc_z": ecc.get(qid, {}).get("ecc_z"), 
            "sem_ent": sement.get(qid, {}).get("semantic_entropy")
        })
    
    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    
    # Write markdown report
    with open(out_md, "w") as f:
        f.write(f"# RAG+UE Report\n\n")
        f.write(f"**Generated:** {datetime.now().isoformat()}\n\n")
        f.write(f"**Model:** {MODEL_ID}\n\n")
        f.write(f"## Summary\n\n")
        f.write(f"- Total: {len(df)}\n")
        f.write(f"- Supported: {(df['correctness']==1).sum()}\n")
        f.write(f"- Not Supported: {(df['correctness']==0).sum()}\n")
        f.write(f"- Unknown: {df['correctness'].isna().sum()}\n\n")
        f.write("## Results\n\n")
        f.write("| ID | Query | SAFE | MARS | ECC_z | SemEnt |\n|--|--|--|--|--|--|\n")
        for _, r in df.iterrows():
            mars_str = format_val(r['mars'])
            ecc_str = format_val(r['ecc_z'])
            sem_str = format_val(r.get('sem_ent'))
            safe_str = str(r['safe']) if pd.notna(r['safe']) else "N/A"
            f.write(f"| {r['id']} | {str(r['query'])[:40]}... | {safe_str} | {mars_str} | {ecc_str} | {sem_str} |\n")
    
    # Correlation analysis
    corr_cols = ["correctness", "mars", "ecc", "ecc_z"]
    if "sem_ent" in df.columns and df["sem_ent"].notna().any():
        corr_cols.append("sem_ent")
    
    corr_df = df[corr_cols].dropna()
    if len(corr_df) > 2:
        correlations = corr_df.corr(method="pearson")
        correlations.to_csv(corr_csv)
        print("\n" + "="*60 + "\nCORRELATION MATRIX\n" + "="*60)
        print(correlations.to_string())
        print("\n" + "="*60)
        print("KEY INSIGHTS:")
        print("="*60)
        if "mars" in correlations.columns:
            print(f"  MARS vs Correctness: {correlations.loc['correctness', 'mars']:.4f}")
        if "ecc" in correlations.columns:
            print(f"  ECC vs Correctness: {correlations.loc['correctness', 'ecc']:.4f}")
        if "sem_ent" in correlations.columns:
            print(f"  Semantic Entropy vs Correctness: {correlations.loc['correctness', 'sem_ent']:.4f}")
    else:
        print(f"Insufficient data for correlations ({len(corr_df)} valid rows)")
    
    banner("DONE")

FINAL_MD = REPORTS_DIR / "notebook.seed1337.report.md"
FINAL_CSV = REPORTS_DIR / "notebook.seed1337.scores.csv"
CORR_CSV = REPORTS_DIR / "notebook.seed1337.correlations.csv"
write_final_report(SAMPLED_QUERIES, ANSWERS_FILE, MARS_JSONL, ECC_JSONL, SAFE_JSONL, SEMENT_JSONL, FINAL_MD, FINAL_CSV, CORR_CSV)

In [ ]:
# ============================================================================
# CELL 16: Cleanup
# ============================================================================
_stop_hb.set()
print("\n" + "="*80 + "\nPIPELINE COMPLETE\n" + "="*80)
print(f"Model: {MODEL_ID}")
print(f"Answers: {ANSWERS_FILE}")
print(f"SAFE: {SAFE_JSONL}")
print(f"MARS: {MARS_JSONL}")
print(f"ECC: {ECC_JSONL}")
print(f"SemEnt: {SEMENT_JSONL}")
print(f"Report: {FINAL_MD}")

---
## Appendix: Semantic Entropy Explanation

### What is Semantic Entropy?

Semantic Entropy is a novel uncertainty estimation method that measures how **semantically diverse** a model's outputs are when asked the same question multiple times with randomness (temperature sampling).

### The Core Idea

**If a model truly knows the answer**, it will produce semantically similar responses even with randomness.

**If a model is uncertain/guessing**, it will produce semantically different responses each time.

### Algorithm Step-by-Step

1. **Generate N answer samples** (default N=5) using temperature sampling
   - Temperature controls randomness: higher T = more random
   - We use T=0.7 (moderate randomness)

2. **Embed all answers** using a sentence transformer
   - Converts text to 384-dimensional vectors
   - Similar meanings → similar vectors

3. **Cluster by semantic similarity**
   - Uses Agglomerative Clustering with cosine distance
   - Threshold=0.85 means answers >85% similar go in same cluster

4. **Compute entropy from cluster distribution**
   - `H = -Σ(p_i × log₂(p_i))`
   - Where `p_i = (size of cluster i) / (total samples)`

5. **Normalize** by max possible entropy: `H_norm = H / log₂(N)`

### Interpretation

| Semantic Entropy | # Clusters | Interpretation |
|------------------|------------|----------------|
| ~0 | 1 | All answers semantically identical → **Confident** |
| ~0.5 | 2-3 | Some variation → **Moderate uncertainty** |
| ~1.0 | 5 (all different) | Every answer different → **High uncertainty** |

### References

- Kuhn et al. (2023). "Semantic Uncertainty: Linguistic Invariances for Uncertainty Estimation in NLG"
- Lin et al. (2023). "LUQ: Long-text Uncertainty Quantification for LLMs"